# 02 — Data Cleaning

## Why this step exists

Raw e-commerce transaction data is never analysis-ready. From Notebook 01 we know this
dataset has: missing Customer IDs, cancelled orders (InvoiceNo starting with 'C'),
and likely some invalid Quantity/Price values (returns, adjustments, errors).

If we don't clean these out, every downstream step is compromised:
- **EDA** would show misleading trends (cancelled orders inflating/deflating totals).
- **Feature engineering** would build features on noise.
- **Model training** would learn patterns that don't reflect real sales behavior.

This notebook uses `cleaning.clean_online_retail()` — the logic lives in `src/` so
it's reusable and testable, not just notebook throwaway code. Here we run it, verify
the effect, and explain each decision.

In [ ]:
import sys
sys.path.append("../src")

from ecommerce_sales_analysis.data_loader import load_raw_data
from ecommerce_sales_analysis.cleaning import clean_online_retail, cleaning_summary

raw = load_raw_data()
cleaned = clean_online_retail(raw)

cleaning_summary(raw, cleaned)

## Verifying each cleaning decision

Let's confirm the cleaning actually did what we intended — never trust, always verify.

In [ ]:
# No missing Customer IDs left
assert cleaned['Customer ID'].isna().sum() == 0 if 'Customer ID' in cleaned.columns else cleaned['CustomerID'].isna().sum() == 0

# No cancelled invoices left
assert not cleaned['InvoiceNo'].str.startswith('C').any()

# No non-positive quantity or price
assert (cleaned['Quantity'] > 0).all()
price_col = 'Price' if 'Price' in cleaned.columns else 'UnitPrice'
assert (cleaned[price_col] > 0).all()

# No duplicate rows
assert cleaned.duplicated().sum() == 0

print('All cleaning checks passed.')

In [ ]:
cleaned[['InvoiceNo', 'StockCode', 'Quantity', price_col, 'Revenue', 'InvoiceDate', 'Country']].head()

In [ ]:
# Save the cleaned dataset for downstream notebooks
from pathlib import Path
processed_path = Path('../data/processed/cleaned_retail.csv')
cleaned.to_csv(processed_path, index=False)
print(f'Saved cleaned data to {processed_path} ({len(cleaned):,} rows)')

## What we learned

_Fill this in after running the cells above:_
- What % of rows were removed, and does that seem reasonable?
- Which cleaning rule removed the most rows?
- Are we comfortable with the tradeoffs (e.g. dropping guest orders with no Customer ID)?

Next: **Notebook 03 — EDA & Visualization**, where we explore trends, seasonality,
and top products/customers/countries in the cleaned data.